In [1]:
import pandas as pd
from feast import FeatureStore

In [2]:
# Feature Store
store = FeatureStore("../03_feature_store/")

M:\disco M\Programacao\PythonProjects\VENVS\env\Lib\site-packages\dask\dataframe\utils.py:274: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  "m": np.timedelta64(1),


In [3]:
store

FeatureStore(
    repo_path=WindowsPath('../03_feature_store'),
    config=RepoConfig(project='ieee_cis_fraud', project_description=None, provider='local', registry_config='data/registry.db', online_config={'type': 'sqlite', 'path': 'data/online_store.db'}, auth={'type': 'no_auth'}, offline_config='dask', batch_engine_config='local', feature_server=None, flags=None, repo_path=WindowsPath('../03_feature_store'), entity_key_serialization_version=3, coerce_tz_aware=True, materialization_config=MaterializationConfig(pull_latest_features=False, online_write_batch_size=None), openlineage_config=None, mlflow_config=None, data_quality_monitoring_config=None),
    registry=not loaded,
    provider=not loaded
)

In [4]:
# Pegando as features
fv = store.get_feature_view(
    "transaction_features"
)

features = [
    f"transaction_features:{field.name}"
    for field in fv.schema
    if field.name not in {
        "transaction_id",
        "transaction_time",
        "label"
    }
]

In [5]:
# dataframe com os ids necessarios
df_entity = pd.read_parquet(
    '../00_dataset/02_final_data/df_train.pq',
    columns=['transaction_id', 'transaction_time', 'label']
)

In [6]:
df_entity.head()

,transaction_id,transaction_time,label
0,2994538,2017-12-03 18:50:58,0
1,2994542,2017-12-03 18:52:10,0
2,2994544,2017-12-03 18:52:29,0
3,2994552,2017-12-03 18:55:49,0
4,2994553,2017-12-03 18:55:53,0


In [7]:
# df training
df_training = store.get_historical_features(
    entity_df=df_entity,
    features=features,
).to_df()

Using transaction_time as the event timestamp. To specify a column explicitly, please name it event_timestamp.


In [8]:
df_training.head()

,transaction_id,transaction_time,label,numerical__id_10,numerical__d14,catgorical__id_35,catgorical__device_info,catgorical__id_31,catgorical__m4,catgorical__product_cd,...,catgorical__id_15,numerical__d12,numerical__d9,catgorical__id_28,catgorical__r_emaildomain,catgorical__device_type,numerical__d7,catgorical__id_38,numerical__id_09,catgorical__id_29
0,2987000,2017-12-02 00:00:00+00:00,0,-999.0,-999.0,0.021117,0.025662,0.021198,0.113490,0.020523,...,0.021117,-999.0,-999.0,0.021127,0.020969,0.021136,-999.0,0.021117,-999.0,0.021127
1,2987001,2017-12-02 00:00:01+00:00,0,-999.0,-999.0,0.020918,0.025479,0.021004,0.037018,0.020350,...,0.020918,-999.0,-999.0,0.020924,0.020748,0.020930,-999.0,0.020918,-999.0,0.020924
2,2987002,2017-12-02 00:01:09+00:00,0,-999.0,-999.0,0.020918,0.025479,0.021004,0.037018,0.020350,...,0.020918,-999.0,-999.0,0.020924,0.020748,0.020930,-999.0,0.020918,-999.0,0.020924
3,2987003,2017-12-02 00:01:39+00:00,0,-999.0,-999.0,0.021117,0.025662,0.021198,0.036351,0.020523,...,0.021117,-999.0,-999.0,0.021127,0.020969,0.021136,-999.0,0.021117,-999.0,0.021127
4,2987004,2017-12-02 00:01:46+00:00,0,-999.0,-999.0,0.045269,0.000000,0.068579,0.018378,0.048630,...,0.048881,-999.0,-999.0,0.052046,0.020748,0.102321,-999.0,0.059340,-999.0,0.051169


# Treinando um Benchmark

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, TunedThresholdClassifierCV
from sklearn.metrics import classification_report

In [10]:
LABEL = 'label'
FEATURES = [
    field.name
    for field in fv.schema
    if field.name not in {
        "transaction_id",
        "transaction_time",
        "label"
    }
]

In [11]:
df_train, df_eval = train_test_split(
    df_training,
    shuffle=True,
    stratify=df_training[LABEL],
    test_size=0.2
)

In [12]:
df_train[LABEL].value_counts(normalize=True)

label
0    0.965011
1    0.034989
Name: proportion, dtype: float64

In [13]:
df_eval[LABEL].value_counts(normalize=True)

label
0    0.965007
1    0.034993
Name: proportion, dtype: float64

In [14]:
X_train, y_train = df_train[FEATURES], df_train[LABEL]
X_eval, y_eval = df_eval[FEATURES], df_eval[LABEL]

In [15]:
# model
model = RandomForestClassifier(n_estimators=100, max_depth=7, min_samples_split=20)

In [16]:
model.fit(X_train, y_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",7
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",20
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [17]:
model_tuned = TunedThresholdClassifierCV(
    RandomForestClassifier(n_estimators=100, max_depth=7, min_samples_split=20),
    scoring="f1"
)

In [18]:
model_tuned.fit(X_train, y_train)

,"estimator estimator: estimator instanceThe classifier, fitted or not, for which we want to optimizethe decision threshold used during `predict`.",RandomForestC...ples_split=20)
,"scoring scoring: str or callable, default=""balanced_accuracy""The objective metric to be optimized. Can be one of:- str: string associated to a scoring function for binary classification, see :ref:`scoring_string_names` for options.- callable: a scorer callable object (e.g., function) with signature ``scorer(estimator, X, y)``. See :ref:`scoring_callable` for details.",'f1'
,"response_method response_method: {""auto"", ""decision_function"", ""predict_proba""}, default=""auto""Methods by the classifier `estimator` corresponding to thedecision function for which we want to find a threshold. It can be:* if `""auto""`, it will try to invoke, for each classifier, `""predict_proba""` or `""decision_function""` in that order.* otherwise, one of `""predict_proba""` or `""decision_function""`. If the method is not implemented by the classifier, it will raise an error.",'auto'
,"thresholds thresholds: int or array-like, default=100The number of decision threshold to use when discretizing the output of theclassifier `method`. Pass an array-like to manually specify the thresholdsto use.",100
,"cv cv: int, float, cross-validation generator, iterable or ""prefit"", default=NoneDetermines the cross-validation splitting strategy to train classifier.Possible inputs for cv are:* `None`, to use the default 5-fold stratified K-fold cross validation;* An integer number, to specify the number of folds in a stratified k-fold;* A float number, to specify a single shuffle split. The floating number should be in (0, 1) and represent the size of the validation set;* An object to be used as a cross-validation generator;* An iterable yielding train, test splits;* `""prefit""`, to bypass the cross-validation.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... warning:: Using `cv=""prefit""` and passing the same dataset for fitting `estimator` and tuning the cut-off point is subject to undesired overfitting. You can refer to :ref:`TunedThresholdClassifierCV_no_cv` for an example. This option should only be used when the set used to fit `estimator` is different from the one used to tune the cut-off point (by calling :meth:`TunedThresholdClassifierCV.fit`).",None
,"refit refit: bool, default=TrueWhether or not to refit the classifier on the entire training set oncethe decision threshold has been found.Note that forcing `refit=False` on cross-validation having morethan a single split will raise an error. Similarly, `refit=True` inconjunction with `cv=""prefit""` will raise an error.",True
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. When `cv` represents across-validation strategy, the fitting and scoring on each data splitis done in parallel. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of cross-validation when `cv` is a float.See :term:`Glossary <random_state>`.",None
,"store_cv_results store_cv_results: bool, default=FalseWhether to store all scores and thresholds computed during the cross-validationprocess.",False
Name,Type,Value
"best_score_ best_score_: float or NoneThe optimal score of the objective metric, evaluated at `best_threshold_`.",float64,0.3183


In [19]:
model_tuned

,"estimator estimator: estimator instanceThe classifier, fitted or not, for which we want to optimizethe decision threshold used during `predict`.",RandomForestC...ples_split=20)
,"scoring scoring: str or callable, default=""balanced_accuracy""The objective metric to be optimized. Can be one of:- str: string associated to a scoring function for binary classification, see :ref:`scoring_string_names` for options.- callable: a scorer callable object (e.g., function) with signature ``scorer(estimator, X, y)``. See :ref:`scoring_callable` for details.",'f1'
,"response_method response_method: {""auto"", ""decision_function"", ""predict_proba""}, default=""auto""Methods by the classifier `estimator` corresponding to thedecision function for which we want to find a threshold. It can be:* if `""auto""`, it will try to invoke, for each classifier, `""predict_proba""` or `""decision_function""` in that order.* otherwise, one of `""predict_proba""` or `""decision_function""`. If the method is not implemented by the classifier, it will raise an error.",'auto'
,"thresholds thresholds: int or array-like, default=100The number of decision threshold to use when discretizing the output of theclassifier `method`. Pass an array-like to manually specify the thresholdsto use.",100
,"cv cv: int, float, cross-validation generator, iterable or ""prefit"", default=NoneDetermines the cross-validation splitting strategy to train classifier.Possible inputs for cv are:* `None`, to use the default 5-fold stratified K-fold cross validation;* An integer number, to specify the number of folds in a stratified k-fold;* A float number, to specify a single shuffle split. The floating number should be in (0, 1) and represent the size of the validation set;* An object to be used as a cross-validation generator;* An iterable yielding train, test splits;* `""prefit""`, to bypass the cross-validation.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... warning:: Using `cv=""prefit""` and passing the same dataset for fitting `estimator` and tuning the cut-off point is subject to undesired overfitting. You can refer to :ref:`TunedThresholdClassifierCV_no_cv` for an example. This option should only be used when the set used to fit `estimator` is different from the one used to tune the cut-off point (by calling :meth:`TunedThresholdClassifierCV.fit`).",None
,"refit refit: bool, default=TrueWhether or not to refit the classifier on the entire training set oncethe decision threshold has been found.Note that forcing `refit=False` on cross-validation having morethan a single split will raise an error. Similarly, `refit=True` inconjunction with `cv=""prefit""` will raise an error.",True
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. When `cv` represents across-validation strategy, the fitting and scoring on each data splitis done in parallel. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of cross-validation when `cv` is a float.See :term:`Glossary <random_state>`.",None
,"store_cv_results store_cv_results: bool, default=FalseWhether to store all scores and thresholds computed during the cross-validationprocess.",False
Name,Type,Value
"best_score_ best_score_: float or NoneThe optimal score of the objective metric, evaluated at `best_threshold_`.",float64,0.3183


In [20]:
print(classification_report(y_eval, model.predict(df_eval[FEATURES])))

              precision    recall  f1-score   support

           0       0.97      1.00      0.98    113975
           1       0.85      0.07      0.12      4133

    accuracy                           0.97    118108
   macro avg       0.91      0.53      0.55    118108
weighted avg       0.96      0.97      0.95    118108



In [21]:
print(classification_report(y_eval, model_tuned.predict(df_eval[FEATURES])))

              precision    recall  f1-score   support

           0       0.97      0.99      0.98    113975
           1       0.38      0.24      0.30      4133

    accuracy                           0.96    118108
   macro avg       0.67      0.61      0.64    118108
weighted avg       0.95      0.96      0.96    118108



# Salvando o modelo

In [22]:
import joblib
from pathlib import Path

In [23]:
ARTFACTS = Path('../artfacts')

In [24]:
joblib.dump(model, ARTFACTS / 'model_mvp_v0.joblib')
joblib.dump(model_tuned, ARTFACTS / 'model_mvp_threshould_tuned_v0.joblib')

['..\\artfacts\\model_mvp_threshould_tuned_v0.joblib']